In [1]:
from typing import List
import numpy as np
import torch
import evaluate
from sklearn.model_selection import train_test_split
import nltk
nltk.download('treebank')

/media/tan/F/AIO2024_hw/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package treebank to /home/tan/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

In [2]:
tagged_sentences = nltk.corpus.treebank.tagged_sents()
print("Number of samples:", len(tagged_sentences))

Number of samples: 3914


In [3]:
sentences, sentences_tags = [], []

for tagged_sentence in tagged_sentences:
    sentence, tags = zip(*tagged_sentence)
    sentences . append ([ word . lower () for word in sentence ])
    sentences_tags.append([tag for tag in tags])

In [4]:
train_sentences, test_sentences, train_tags, test_tags = train_test_split(
    sentences,
    sentences_tags,
    test_size = 0.3
)

valid_sentences, test_sentences, valid_tags, test_tags = train_test_split(
    test_sentences,
    test_tags,
    test_size = 0.5
)

In [6]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
from torch.utils.data import Dataset

model_name = "QCRI/bert-base-multilingual-cased-pos-english"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast = True
)

model = AutoModelForTokenClassification.from_pretrained(model_name)

MAX_LEN = 256

class PosTaggingDataset(Dataset):
    def __init__(
        self,
        sentences: List[List[str]],
        tags: List[List[str]],
        tokenizer,
        label2id: dict,
        max_len: int = MAX_LEN,
    ):
        """
        Dataset cho bài toán POS tagging.

        Args:
            sentences (List[List[str]]): Danh sách câu, mỗi câu là một danh sách token.
            tags (List[List[str]]): Danh sách nhãn, mỗi câu là một danh sách nhãn tương ứng với từng token.
            tokenizer: Tokenizer từ Hugging Face Transformers.
            label2id (dict): Dictionary ánh xạ nhãn thành ID.
            max_len (int, optional): Độ dài tối đa của mỗi câu. Mặc định là 256.
        """
        super().__init__()
        self.sentences = sentences
        self.tags = tags
        self.tokenizer = tokenizer
        self.label2id = label2id
        self.max_len = max_len

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        input_tokens = self.sentences[idx]
        label_tokens = self.tags[idx]

        # Chuyển token thành ID
        input_ids = self.tokenizer.convert_tokens_to_ids(input_tokens)
        attention_mask = [1] * len(input_ids)
        labels = [self.label2id[token] for token in label_tokens]

        return {
            "input_ids": self.pad_and_truncate(input_ids, pad_id=self.tokenizer.pad_token_id),
            "labels": self.pad_and_truncate(labels, pad_id=self.label2id["0"]),
            "attention_mask": self.pad_and_truncate(attention_mask, pad_id=0),
        }

    def pad_and_truncate(self, inputs: List[int], pad_id: int):
        """
        Thêm padding hoặc cắt bớt chuỗi đầu vào để đảm bảo độ dài cố định.

        Args:
            inputs (List[int]): Chuỗi đầu vào (ID của tokens hoặc labels).
            pad_id (int): Giá trị ID dùng để padding.

        Returns:
            torch.Tensor: Tensor đã được padding/truncate.
        """
        if len(inputs) < self.max_len:
            padded_inputs = inputs + [pad_id] * (self.max_len - len(inputs))
        else:
            padded_inputs = inputs[: self.max_len]

        return torch.tensor(padded_inputs, dtype=torch.long)

Some weights of the model checkpoint at QCRI/bert-base-multilingual-cased-pos-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [8]:
from collections import defaultdict

label2id = defaultdict(int, model.config.label2id)
id2label = {id: label for label, id in label2id.items()}

In [10]:
label2id

defaultdict(int,
            {'#': 7,
             '$': 6,
             "''": 5,
             ',': 2,
             '-LRB-': 17,
             '-RRB-': 32,
             '.': 4,
             ':': 3,
             'CC': 8,
             'CD': 9,
             'DT': 10,
             'EX': 11,
             'FW': 12,
             'IN': 13,
             'JJ': 14,
             'JJR': 15,
             'JJS': 16,
             'LS': 18,
             'MD': 19,
             'NN': 20,
             'NNP': 21,
             'NNPS': 22,
             'NNS': 23,
             'O': 0,
             'PDT': 24,
             'POS': 25,
             'PRP': 26,
             'PRP$': 27,
             'RB': 28,
             'RBR': 29,
             'RBS': 30,
             'RP': 31,
             'SYM': 33,
             'TO': 34,
             'UH': 35,
             'VB': 36,
             'VBD': 37,
             'VBG': 38,
             'VBN': 39,
             'VBP': 40,
             'VBZ': 41,
             'WDT': 42,
      

In [9]:
train_dataset = PosTaggingDataset(train_sentences, train_tags, tokenizer, label2id)
val_dataset = PosTaggingDataset(valid_sentences, valid_tags, tokenizer, label2id)
test_sentences = PosTaggingDataset(test_sentences,test_tags,tokenizer,label2id)

In [11]:
accuracy = evaluate.load("accuracy")

ignore_label = len(label2id)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    mask = labels != ignore_label
    predictions = np.argmax(predictions, axis = -1)
    return accuracy.compute(
        predictions = predictions[mask],
        references = labels[mask]
    )

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "out_dir",
    learning_rate = 1e-5,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    num_train_epochs = 10,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    tokenizer = tokenizer,
    compute_metrics = compute_metrics
)